# A6 continue-train from hybrid LoRA (Colab A100)

**Goal:** start from your already-trained hybrid adapter, then train further on **new ChatGPT JSONL** (not a full retrain of the old 7.8k hybrid pack).

## Drive layout (create once)

```
MyDrive/usu-a6/
  adapters/hybrid_start/          # copy of local train_hybrid/best
  chatgpt_continue_train.jsonl    # your new approved rows
  chatgpt_continue_dev.jsonl      # small held-out set (20–50+ rows)
  gold_answers.json               # optional; keyword scoring after train
```

## JSONL schema (every row)

```json
{"example_id": "chatgpt:skill:001", "instruction": "...", "response": "...", "category": "letter_counting", "source": "chatgpt:skill", "review_status": "approved"}
```

- `review_status` **must** be `approved`
- Never paste graded homework stems (`prepare-qa` will reject them)
- See `notebooks/chatgpt_continue_example.jsonl` for safe examples

## Runtime / A100 throughput

Browser Colab → **A100 80GB** → run top → bottom.

Default train knobs use a **large micro-batch** (`batch_size=64`, `accumulation=1`) so the GPU is fed in parallel instead of the old Quadro-style `1×8` (~3 GiB). Tune `BATCH_SIZE` in the config cell if you OOM or still sit under ~20 GiB.

In [ ]:
# 0) Colab kernel check
import os
import sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    pass

assert IN_COLAB, "Select a Colab kernel (A100), then re-run."
print("Connected to Colab OK")

In [ ]:
# 1) GPU check
import subprocess

import torch

print(subprocess.getoutput("nvidia-smi"))
assert torch.cuda.is_available(), "No CUDA — pick a Colab GPU runtime"
print("GPU=", torch.cuda.get_device_name(0))

In [ ]:
# 2) Mount Drive, clone repo, copy adapter + new JSONL into the repo
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/usu-a6")
INIT_ADAPTER_SRC = (
    DRIVE / "adapters" / "hybrid_start"
)  # upload local train_hybrid/best here
TRAIN_JSONL_SRC = DRIVE / "chatgpt_continue_train.jsonl"
DEV_JSONL_SRC = DRIVE / "chatgpt_continue_dev.jsonl"
GOLD_SRC = DRIVE / "gold_answers.json"  # optional

REPO_URL = "https://github.com/IMJONEZZ/USU-LLM-Class-F26.git"
BRANCH = "talha_shahzad_6"
REPO = Path("/content/USU-LLM-Class-F26")

for pth, label in [
    (INIT_ADAPTER_SRC, "init adapter"),
    (TRAIN_JSONL_SRC, "train jsonl"),
    (DEV_JSONL_SRC, "dev jsonl"),
]:
    assert pth.exists(), f"Missing {label}: {pth}"

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.check_call(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO)]
)

QA = REPO / "data" / "assignment6" / "qa"
QA.mkdir(parents=True, exist_ok=True)
ADAPTER_DST = REPO / "data" / "assignment6" / "adapters" / "hybrid_start"
if ADAPTER_DST.exists():
    shutil.rmtree(ADAPTER_DST)
shutil.copytree(INIT_ADAPTER_SRC, ADAPTER_DST)
shutil.copy(TRAIN_JSONL_SRC, QA / "chatgpt_continue_train.jsonl")
shutil.copy(DEV_JSONL_SRC, QA / "chatgpt_continue_dev.jsonl")
if GOLD_SRC.is_file():
    shutil.copy(GOLD_SRC, REPO / "data" / "assignment6" / "gold_answers.json")

%cd /content/USU-LLM-Class-F26
!ls -la data/assignment6/adapters/hybrid_start | head
!wc -l data/assignment6/qa/chatgpt_continue_*.jsonl

In [ ]:
# 3) Install train deps (same pins as the other Colab notebook)
%pip -q install -U pip
%pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip -q install peft trl accelerate bitsandbytes datasets "transformers==4.57.6"
%pip -q install evaluate rouge-score bert-score

In [ ]:
# 4) Hugging Face login (gated Llama Instruct)
import getpass
import os

from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF token: ")
login(token=token)
os.environ["HF_TOKEN"] = token

In [ ]:
# 5) Quick validate new JSONL + prepare-qa (new data only)
%cd /content/USU-LLM-Class-F26
import json
from pathlib import Path

from src.trainer import find_graded_leakage, validate_qa_record

train_path = Path("data/assignment6/qa/chatgpt_continue_train.jsonl")
dev_path = Path("data/assignment6/qa/chatgpt_continue_dev.jsonl")


def load_jsonl(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


train_rows = load_jsonl(train_path)
dev_rows = load_jsonl(dev_path)
assert train_rows and dev_rows, "train/dev JSONL must be nonempty"
for row in train_rows + dev_rows:
    validate_qa_record(row)
    assert row["review_status"] == "approved", row["example_id"]

leak = find_graded_leakage(train_rows + dev_rows)
assert not leak, f"Graded-stem leakage — fix JSONL first: {leak[:3]}"
print(f"OK train={len(train_rows)} dev={len(dev_rows)} (no graded leakage)")

!rm -rf data/assignment6/prepared_continue
!python -m src.trainer prepare-qa \
  --train data/assignment6/qa/chatgpt_continue_train.jsonl \
  --dev data/assignment6/qa/chatgpt_continue_dev.jsonl \
  --output data/assignment6/prepared_continue

In [ ]:
# 5b) A100 train knobs — use the GPU
# Quadro-era defaults were batch=1, accum=8 (~3 GiB). A100 80GB can take much larger micro-batches.
BATCH_SIZE = 128          # drop to 64 if OOM; try 256 if still <~40 GiB after a few steps
ACCUMULATION = 1         # effective batch = BATCH_SIZE * ACCUMULATION
EPOCHS = 3
FEASIBILITY_STEPS = 4   # short smoke test with the SAME batch/accum as train
print(f"effective_batch={BATCH_SIZE * ACCUMULATION} epochs={EPOCHS}")


In [ ]:
# 6) Feasibility + continue-train (large A100 batches)
# Requires branch with --init-adapter. Feasibility options MUST match train.
%cd /content/USU-LLM-Class-F26
from pathlib import Path
!rm -rf data/assignment6/feasibility_continue data/assignment6/train_continue

import subprocess

common = [
    "--prepared", "data/assignment6/prepared_continue",
    "--init-adapter", "data/assignment6/adapters/hybrid_start",
    "--batch-size", str(BATCH_SIZE),
    "--accumulation", str(ACCUMULATION),
    "--learning-rate", "2e-4",
    "--rank", "4",
    "--alpha", "8",
    "--targets", "q_proj,v_proj",
    "--no-gradient-checkpointing",
]

subprocess.check_call(
    ["python", "-m", "src.trainer", "feasibility", *common,
     "--output", "data/assignment6/feasibility_continue",
     "--steps", str(FEASIBILITY_STEPS)]
)
subprocess.check_call(
    ["python", "-m", "src.trainer", "train", *common,
     "--feasibility", "data/assignment6/feasibility_continue",
     "--output", "data/assignment6/train_continue",
     "--epochs", str(EPOCHS)]
)
assert Path("data/assignment6/train_continue/success.json").is_file()
print("TRAIN_OK")
!nvidia-smi
!ls data/assignment6/train_continue/best | head


In [ ]:
# 7) Generate graded-10 answers with the continued adapter
%cd /content/USU-LLM-Class-F26
from datetime import UTC, datetime
from pathlib import Path

import json

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from src.trainer import GRADED_QUESTIONS

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = Path("data/assignment6/train_continue/best")
OUT = Path("data/assignment6/after_continue_train.json")
DEVICE = "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(base, ADAPTER)
model.to(DEVICE)
model.eval()


def ask(question: str) -> str:
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[0, inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


examples = []
for q in GRADED_QUESTIONS:
    ans = ask(q)
    examples.append({"question": q, "answer": ans})
    print("Q:", q)
    print("A:", ans)
    print("---")

payload = {
    "model": MODEL_ID,
    "adapter": str(ADAPTER),
    "device": DEVICE,
    "max_new_tokens": 128,
    "generation": "greedy",
    "created_utc": datetime.now(UTC).isoformat(),
    "examples": examples,
}
OUT.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
print("Wrote", OUT)

In [ ]:
# 8) Optional keyword score vs gold_answers.json + save adapter to Drive
%cd /content/USU-LLM-Class-F26
import json
import shutil
from pathlib import Path

DATA = Path("data/assignment6")
gold_path = DATA / "gold_answers.json"
pred = json.loads((DATA / "after_continue_train.json").read_text())


def norm(text: str) -> str:
    return " ".join(str(text).lower().split())


def keyword_pass(prediction: str, item: dict) -> bool:
    if any(norm(bad) in norm(prediction) for bad in item.get("reject_if") or []):
        return False
    keywords = item.get("keywords") or []
    hits = [kw for kw in keywords if norm(kw) in norm(prediction)]
    if item.get("match") == "all":
        return len(hits) == len(keywords)
    return bool(hits)


if gold_path.is_file():
    gold = json.loads(gold_path.read_text())
    passed = 0
    for item, ex in zip(gold, pred["examples"], strict=True):
        ok = keyword_pass(ex["answer"], item)
        passed += int(ok)
        print(f"Q{item['id']}: {'PASS' if ok else 'FAIL'} | {ex['answer'][:80]}")
    print(f"Keyword {passed}/{len(gold)}")
else:
    print(
        "No gold_answers.json on Drive — skipped scoring. Answers still in after_continue_train.json"
    )

dst = Path("/content/drive/MyDrive/usu-a6/adapters/hybrid_continue_chatgpt")
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(DATA / "train_continue" / "best", dst)
shutil.copy(DATA / "after_continue_train.json", dst / "after_continue_train.json")
print("Saved adapter + answers →", dst)